In [1]:
pip install iterative-stratification

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import gc
import sys
import time
import math
import json
import random
import numpy as np
import pandas as pd
import typing as tp
from tqdm.auto import tqdm

import seaborn as sns
import matplotlib.pyplot as plt

import ast
from pathlib import Path
from itertools import islice
from collections import Counter, defaultdict

import wave
import soundfile

import librosa
import soundfile as sf
from IPython.display import Audio, display

from scipy.sparse import coo_matrix
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

import glob
import warnings
warnings.filterwarnings("ignore")


In [3]:
class CFG():
    SEED          = 42
    N_FOLDS       = 5
    base_dir      = "/kaggle/input/competitions/birdclef-2026"
    data_dir      = "/kaggle/input/datasets/tatsuyayamamoto/bird-2026-processed-ver1/dataframe/preprocessing_df.csv"
    # Librosa
    FS            = 32_000
    N_FFT         = 1_024
    HOP_LEN       = 512
    N_MELS        = 128
    FMIN          = 50
    FMAX          = 14_000
    DURATION      = 5
    # other
    debug         = False
cfg = CFG()
print(f"Debug : {cfg.debug}")


Debug : False


### SEED Everything

In [4]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    print(f"SEED is {seed}")
    
seed_everything(cfg.SEED)

SEED is 42


### Make Directry

In [5]:
DATA = "./data"
if not os.path.exists(DATA):
    os.makedirs(DATA)

AUDIO = "./audio"
if not os.path.exists(AUDIO):
    os.makedirs(AUDIO)

### Data

In [6]:
taxonomy        = pd.read_csv(os.path.join(cfg.base_dir, "taxonomy.csv"))
submission      = pd.read_csv(os.path.join(cfg.base_dir, "sample_submission.csv"))
print(f"Taxonomy Shape : {taxonomy.shape}")
print(f"Sub      Shape : {submission.shape}")

Taxonomy Shape : (234, 5)
Sub      Shape : (3, 235)


In [7]:
submission

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


### label2idx & idx2label

In [8]:
CLASSES = taxonomy["primary_label"].values.tolist()
print(f"{len(CLASSES)} Species")

label2idx = {label: idx for idx, label in enumerate(taxonomy["primary_label"].values)}
idx2label = {idx: label for label, idx in label2idx.items()}

print(f"label2idx : {dict(islice(label2idx.items(), 3))}")
print(f"idx2label : {dict(islice(idx2label.items(), 3))}")

sub_labels = submission.columns[1:].tolist()
print(f"Submission: {len(sub_labels)}")
print(list(label2idx.keys()) == sub_labels)

234 Species
label2idx : {'1161364': 0, '116570': 1, '1176823': 2}
idx2label : {0: '1161364', 1: '116570', 2: '1176823'}
Submission: 234
True


In [9]:
def safe_literal_eval(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

def load_df(path):
    df = pd.read_csv(path)
    df["labels"] = df["labels"].apply(safe_literal_eval)
    return df

### All Data

In [10]:
all_df = load_df(cfg.data_dir)
print(f"All Data Shape  : {all_df.shape}")
print(f"Labels Type     : {type(all_df['labels'].iloc[0])}")
print(f"Species number  : {all_df['labels'].explode().nunique()}")
print(f'Train Species   : {all_df.loc[all_df["is_ss"]==0, "labels"].explode().nunique()}')
print(f'SS    Species   : {all_df.loc[all_df["is_ss"]==1, "labels"].explode().nunique()}')
print(f"Tran and SS \n {all_df['is_ss'].value_counts()}")
display(all_df.head(2))
print(len(set(all_df.loc[all_df["is_ss"] == 0, "labels"].explode().unique()).intersection(all_df.loc[all_df["is_ss"]==1, "labels"].explode().unique())))

All Data Shape  : (35804, 10)
Labels Type     : <class 'list'>
Species number  : 234
Train Species   : 206
SS    Species   : 75
Tran and SS 
 is_ss
0    35379
1      425
Name: count, dtype: int64


,filename,audio_id,labels,class_name,is_ss,weight,duration,n_labels,group_id,class_count
0,1161364/iNat1114648.ogg,1161364/iNat1114648,[1161364],Insecta,0,1.0,28.3,1,1161364/iNat1114648,212
1,1161364/iNat1216197.ogg,1161364/iNat1216197,[1161364],Insecta,0,1.0,18.0,1,1161364/iNat1216197,212


47


In [11]:
print(f"All DF filename Nunique: {all_df['filename'].nunique():,}")
print(f"All DF is_ss==0[train] : {all_df.loc[all_df['is_ss']==0, 'filename'].nunique():,}")
print(f"All DF is_ss==1[ss]    : {all_df.loc[all_df['is_ss']==1, 'filename'].nunique():,}")
print()
print(f"is_ss==0[train] audio id: {all_df.loc[all_df['is_ss']==0, 'audio_id'].nunique():,}")
print(f"is_ss==1[ss]    audio id: {all_df.loc[all_df['is_ss']==1, 'audio_id'].nunique():,}")
print()
print(f"is_ss==0[train] group id: {all_df.loc[all_df['is_ss']==0, 'group_id'].nunique():,}")
print(f"is_ss==1[ss]    group id: {all_df.loc[all_df['is_ss']==1, 'group_id'].nunique():,}")
print()
print(f"is_ss==0[train] class   : \n{all_df.loc[all_df['is_ss']==0, 'class_name'].unique()}")
print()
print(f"is_ss==1[ss] class      : \n{all_df.loc[all_df['is_ss']==1, 'class_name'].unique()}")

All DF filename Nunique: 35,245
All DF is_ss==0[train] : 35,179
All DF is_ss==1[ss]    : 66

is_ss==0[train] audio id: 35,374
is_ss==1[ss]    audio id: 425

is_ss==0[train] group id: 35,183
is_ss==1[ss]    group id: 78

is_ss==0[train] class   : 
['Insecta' 'Reptilia' 'Amphibia' 'Mammalia' 'Amphibia,Aves' 'Aves'
 'Aves,Mammalia']

is_ss==1[ss] class      : 
['Insecta' 'Insecta,Reptilia' 'Aves,Insecta' 'Aves,Insecta,Reptilia'
 'Mammalia' 'Aves' 'Aves,Mammalia' 'Amphibia' 'Amphibia,Aves,Mammalia'
 'Amphibia,Aves' 'Amphibia,Mammalia' 'Amphibia,Insecta'
 'Amphibia,Aves,Insecta' 'Insecta,Mammalia' 'Insecta,Mammalia,Reptilia']


In [12]:
all_df["class_name"].value_counts()

class_name
Aves                         34667
Amphibia                       622
Insecta                        212
Mammalia                        95
Amphibia,Aves                   86
Aves,Insecta                    50
Amphibia,Insecta                19
Amphibia,Aves,Mammalia          11
Insecta,Reptilia                 6
Aves,Insecta,Reptilia            6
Reptilia                         5
Aves,Mammalia                    5
Amphibia,Mammalia                5
Amphibia,Aves,Insecta            5
Insecta,Mammalia                 5
Insecta,Mammalia,Reptilia        5
Name: count, dtype: int64

### Fold

In [13]:
train_only = all_df[all_df["is_ss"] == 0].reset_index(drop=True)
ss_only    = all_df[all_df["is_ss"] == 1].reset_index(drop=True)
print(f"Train Only : {train_only.shape}")
print(f"SS    Only : {ss_only.shape}")

Train Only : (35379, 10)
SS    Only : (425, 10)


### Label

In [14]:
def make_labels_array(df):
    labels_array = np.zeros((len(df), len(label2idx)), dtype=np.float32)
    
    for idx, labels in enumerate(df["labels"].values):
        for l in labels:
            labels_array[idx, label2idx[l]] = 1
    print(f"Labels Array : {labels_array.shape}")
    return labels_array

In [15]:
train_only_labels_arr = make_labels_array(train_only)
ss_only_labels_arr    = make_labels_array(ss_only)

Labels Array : (35379, 234)
Labels Array : (425, 234)


In [16]:
# labels: (N, num_classes)
# Use train_only_labels_arr

train_only["idx"] = np.arange(len(train_only))

grouped = train_only.groupby("group_id")["idx"].apply(list)

group_ids    = []
group_labels = []

for gid, indices in grouped.items():
    group_ids.append(gid)
    
    # group内のlabelをOR（max）でまとめる
    labels      = train_only_labels_arr[indices]
    group_label = labels.max(axis=0)
    
    group_labels.append(group_label)

group_labels = np.stack(group_labels)
group_df = pd.DataFrame({
    "group_id": group_ids
})
print(f"Group df : {group_df.shape}")

Group df : (35183, 1)


In [17]:
#iterstrat supports multiple labels
mskf = MultilabelStratifiedKFold(
    n_splits=cfg.N_FOLDS,
    shuffle=True,
    random_state=cfg.SEED
)

group_df["fold"] = -1

for fold_id, (_, val_idx) in enumerate(mskf.split(group_df, group_labels)):
    group_df.loc[val_idx, "fold"] = fold_id

In [18]:
train_only = train_only.merge(
    group_df[["group_id", "fold"]],
    on="group_id",
    how="left"
)

In [19]:
train_only.head(2)

,filename,audio_id,labels,class_name,is_ss,weight,duration,n_labels,group_id,class_count,idx,fold
0,1161364/iNat1114648.ogg,1161364/iNat1114648,[1161364],Insecta,0,1.0,28.3,1,1161364/iNat1114648,212,0,4
1,1161364/iNat1216197.ogg,1161364/iNat1216197,[1161364],Insecta,0,1.0,18.0,1,1161364/iNat1216197,212,1,0


In [20]:
print(train_only.groupby("group_id")["fold"].nunique().value_counts())
print()
print(train_only["fold"].value_counts().sort_index())

fold
1    35183
Name: count, dtype: int64

fold
0    6989
1    7155
2    7028
3    7125
4    7082
Name: count, dtype: int64


In [21]:
print(train_only.loc[(train_only["is_ss"] == 0)&(train_only["duration"]>=300)&(train_only["duration"]<600)].shape[0])
print(train_only.loc[(train_only["is_ss"] == 0)&(train_only["duration"]>=600)].shape[0])


138
234


In [22]:
train_only.loc[(train_only["is_ss"] == 0)&(train_only["duration"]>=300)&(train_only["duration"]<600)].head()

,filename,audio_id,labels,class_name,is_ss,weight,duration,n_labels,group_id,class_count,idx,fold
35007,74113/XC1066320.ogg,74113/XC1066320,[74113],Mammalia,0,1.000000,318.6,1,74113/XC1066320,95,35007,2
35008,baffal1/XC530447.ogg,baffal1/XC530447,[baffal1],Aves,0,1.000000,461.8,1,baffal1/XC530447,34471,35008,1
35009,baffal1/XC367718.ogg,baffal1/XC367718,[baffal1],Aves,0,1.000000,424.6,1,baffal1/XC367718,34471,35009,1
35010,banana/XC214521.ogg,banana/XC214521,[banana],Aves,0,1.000000,569.3,1,banana/XC214521,34471,35010,3
35011,batbel1/XC669414.ogg,batbel1/XC669414,"[batbel1, rufgna3]",Aves,0,0.707107,504.5,2,batbel1/XC669414,34471,35011,3


In [23]:
train_only.loc[(train_only["is_ss"] == 0)&(train_only["duration"]>=600)].head(3)

,filename,audio_id,labels,class_name,is_ss,weight,duration,n_labels,group_id,class_count,idx,fold
35145,baffal1/XC406518.ogg,baffal1/XC406518_0-300,[baffal1],Aves,0,0.7,600.0,1,baffal1/XC406518,34471,35145,3
35146,baffal1/XC406518.ogg,baffal1/XC406518_240-540,[baffal1],Aves,0,0.7,600.0,1,baffal1/XC406518,34471,35146,3
35147,baffal1/XC406518.ogg,baffal1/XC406518_300-600,[baffal1],Aves,0,0.3,600.0,1,baffal1/XC406518,34471,35147,3


In [24]:
for f in range(cfg.N_FOLDS):
    t = train_only[train_only["fold"] != f].reset_index(drop=True)
    v = train_only[train_only["fold"] == f].reset_index(drop=True)
    s = ss_only.copy()
    c = pd.concat([t, v],axis=0, ignore_index=True)
    print()
    print(f"Fold: {f}")
    print(f"Train length : {len(t):,}")
    print(f"Valid length : {len(v):,}")
    print()
    print(f"SS    length : {len(s):,}")
    print(f"Training     : {len(c):,}")
    print()
    print(c['class_name'].value_counts())
    t_labels = t["labels"].explode().unique()
    ss_labels= s["labels"].explode().unique()
    v_labels = v["labels"].explode().unique()
    t_ss_labe= set(t_labels.tolist()+ss_labels.tolist())
    print()
    print(f"Train Labels Nunique: {len(t_labels)}")
    print(f"SS    Labels Nunique: {len(ss_labels)}")
    print(f"Valid Labels Nunique: {len(v_labels)}")
    print(f"Train & SS Labels NU: {len(t_ss_labe)}")
    print(f"Valid in All Labels : {len(set(v_labels).intersection(t_ss_labe))}")
    print(f"Difference          : {list(set(v_labels).difference(t_ss_labe))}")
    print()
    print(f"Train Group id      : {t['group_id'].nunique():,}")
    print(f"Valid Group id      : {v['group_id'].nunique():,}")
    print(f"Training            : {c['group_id'].nunique():,}")
    print()
    print(f"Training Duration >=300 : {c[c['duration']>=300].shape[0]}")
    print(f"Training Duration < 300 : {c[c['duration']<300].shape[0]}")
    print(f"valid    Duration >=300 : {v[v['duration']>=300].shape[0]}")
    print(f"valid    Duration < 300 : {v[v['duration']<300].shape[0]}")
    print()
    print(f"Training Nlabels \n{c['n_labels'].value_counts()}")
    print(f"Valid    Nlabels \n{v['n_labels'].value_counts()}")
    print(f"{'+*'*20}")


Fold: 0
Train length : 28,390
Valid length : 6,989

SS    length : 425
Training     : 35,379

class_name
Aves             34654
Amphibia           427
Insecta            195
Mammalia            90
Reptilia             5
Amphibia,Aves        5
Aves,Mammalia        3
Name: count, dtype: int64

Train Labels Nunique: 205
SS    Labels Nunique: 75
Valid Labels Nunique: 199
Train & SS Labels NU: 233
Valid in All Labels : 198
Difference          : ['209233']

Train Group id      : 28,220
Valid Group id      : 6,963
Training            : 35,183

Training Duration >=300 : 372
Training Duration < 300 : 35007
valid    Duration >=300 : 54
valid    Duration < 300 : 6935

Training Nlabels 
n_labels
1     30951
2      2650
3      1066
4       382
5       168
6        87
7        35
9        12
8        11
11        7
13        5
16        4
10        1
Name: count, dtype: int64
Valid    Nlabels 
n_labels
1    6082
2     535
3     214
4      79
5      40
6      20
7      10
9       6
8       3
Name: c

### DataFrame Save

In [25]:
train_only.to_csv(os.path.join(DATA, "train.csv"), index=False)
ss_only.to_csv(os.path.join(DATA, "ss.csv"), index=False)

In [26]:
with open(os.path.join(DATA, "idx2label.json"), mode="w") as f:
    json.dump(idx2label, f)

with open(os.path.join(DATA, "label2idx.json"), mode="w") as f:
    json.dump(label2idx, f)

In [27]:
with open(os.path.join(DATA, "idx2label.json"), mode="r") as f:
    i2l = json.load(f)

i2l = {int(k):v for k, v in i2l.items()}
i2l == idx2label

True